#### **Neural Style Transfer**

Neural style transfer consists of applying the style of a reference image to a target image while conserving the content of the target image (Leon A. Gatys, Alexander S. Ecker, and Matthias Bethge, “A Neural Algorithm of Artistic Style,” arXiv (2015), https://arxiv.org/abs/1508.06576.)

In this context, `style` essentially means textures, colors, and visual patterns in the image, at various spatial scales; and the `content` is the higher-level macrostructure of the image. So we are going to define a loss function conserving the content of the original image while adopting the style of the reference image.

In [ ]:
### Defining initial variables
import tensorflow as tf

# The original notebook uses Keras graph APIs; disable eager mode for compatibility.
if tf.executing_eagerly():
    tf.compat.v1.disable_eager_execution()

from tensorflow.keras.utils import load_img, img_to_array

target_image_path = tf.keras.utils.get_file('dog.jpg', 'https://storage.googleapis.com/download.tensorflow.org/example_images/YellowLabradorLooking_new.jpg')
style_reference_image_path = tf.keras.utils.get_file('style_reference.jpg', 'https://storage.googleapis.com/download.tensorflow.org/example_images/Vassily_Kandinsky%2C_1913_-_Composition_7.jpg')


# Dimensions of the generated picture.
width, height = load_img(target_image_path).size
img_height = 400
img_width = int(width * img_height / height)

195196/195196 [==============================] - 0s 1us/step


We need some auxiliary functions for loading, preprocessing, and postprocessing the images that go in and out of the VGG19 convnet.

In [39]:
### Auxliary functions

import numpy as np
from tensorflow.keras.applications import vgg19

def evaluate_tensor(value):
    if hasattr(value, 'numpy'):
        return value.numpy()
    if isinstance(value, tf.Tensor):
        with tf.compat.v1.Session() as sess:
            return sess.run(value)
    return value


def preprocess_image(image_path):
    img = load_img(image_path, target_size=(img_height, img_width))
    img = img_to_array(img)
    img = np.expand_dims(img, axis=0)
    img = vgg19.preprocess_input(img)
    return tf.convert_to_tensor(img, dtype=tf.float32)

def deprocess_image(x):
    x = evaluate_tensor(x)
    x = np.array(x, dtype=np.float32)
    if x.ndim == 1:
        x = x.reshape((img_height, img_width, 3))
    elif len(x.shape) == 4:
        x = np.squeeze(x, 0)
    assert len(x.shape) == 3
    if x.shape[2] == 4:
        x = x[:, :, :3]

    # Zero-center by mean pixel
    x[:, :, 0] += 103.939
    x[:, :, 1] += 116.779
    x[:, :, 2] += 123.68
    x = x[:, :, ::-1]
    x = np.clip(x, 0, 255).astype('uint8')
    return x

Let’s set up the VGG19 network. It takes as input a batch of three images: the style-reference image, the target image, and a placeholder that will contain the generated
image. A placeholder is a symbolic tensor, the values of which are provided externally via Numpy arrays.

In [40]:
### Loading the pretrained VGG19 model and applying it to the three images
import tensorflow as tf
from tensorflow.keras import backend as K

if tf.executing_eagerly():
    tf.compat.v1.disable_eager_execution()

target_image = preprocess_image(target_image_path)
style_reference_image = preprocess_image(style_reference_image_path)
combination_image = tf.compat.v1.placeholder(shape=(1, img_height, img_width, 3), dtype=tf.float32)

input_tensor = tf.concat([target_image, style_reference_image, combination_image], axis=0) # Combine the 3 images into a single Keras tensor

print('Model loaded.')

model = vgg19.VGG19(input_tensor=input_tensor, weights='imagenet', include_top=False)

Model loaded.


#### **The content loss**

Activations from earlier layers in a network contain local information about the image, whereas activations from higher layers contain increasingly global, abstract information. A good candidate for content loss is thus the L2 norm between the activations of an upper layer in a pretrained convnet, computed over the target image, and the activations of the same layer computed over the generated image. This guarantees that, as seen from the upper layer, the generated image will look similar to the original target image. 

In [41]:
from tensorflow.keras import backend as K

def content_loss(base, combination):
    return K.sum(K.square(combination - base))

#### **The style loss**

For the style loss, Gatys et al. use the Gram matrix of a layer’s activations: the inner product of the feature maps of a given layer. This inner product can be understood as representing a map of the correlations between the layer’s features. These feature correlations capture the statistics of the patterns of a particular spatial scale, which empirically correspond to the appearance of the textures found at this scale.

In [42]:
def gram_matrix(x):
    features = K.batch_flatten(K.permute_dimensions(x, (2, 0, 1)))
    gram = K.dot(features, K.transpose(features))
    return gram

In [43]:
def style_loss(style, combination):
    S = gram_matrix(style)
    C = gram_matrix(combination)
    channels = 3
    size = img_height * img_width
    return K.sum(K.square(S - C)) / (4. * (channels ** 2) * (size ** 2))

To these two loss components, you add a third: the total variation loss, which operates on the pixels of the generated combination image. It encourages spatial continuity in
the generated image, thus avoiding overly pixelated results.

#### **Total variation loss**

In [44]:
def total_variation_loss(x):
    a = K.square(
        x[:, :img_height - 1, :img_width - 1, :] - x[:, 1:, :img_width - 1, :])
    b = K.square(
        x[:, :img_height - 1, :img_width - 1, :] - x[:, :img_height - 1, 1:, :])
    return K.sum(K.pow(a + b, 1.25))

#### **Defining the final loss that we'll minimize**

In [45]:
outputs_dict = dict([(layer.name, layer.output) for layer in model.layers]) # Dict mapping layer names to activation tensors

content_layer = 'block5_conv2' # Layer to use for content loss
style_layers = ['block1_conv1', 'block2_conv1', 'block3_conv1', 'block4_conv1', 'block5_conv1'] # Layers to use for style loss

total_variation_weight = 1e-4
style_weight = 1.0
content_weight = 0.025

loss = K.variable(0.) # Initialize the loss variable
layer_features = outputs_dict[content_layer] # Get the features of the content layer
target_image_features = layer_features[0, :, :, :] # Get the features of the target image
combination_features = layer_features[2, :, :, :] # Get the features of the combination image
loss = loss + content_weight * content_loss(target_image_features, combination_features) # Add content loss to the total loss

for layer_name in style_layers: # Add style loss for each layer
    layer_features = outputs_dict[layer_name]
    style_reference_features = layer_features[1, :, :, :]
    combination_features = layer_features[2, :, :, :]
    sl = style_loss(style_reference_features, combination_features)
    loss = loss + (style_weight / len(style_layers)) * sl

loss = loss + total_variation_weight * total_variation_loss(combination_image) # Add total variation loss to the total loss

Finally, you’ll set up the gradient-descent process. In the original Gatys et al. paper, optimization is performed using the L-BFGS algorithm. However, It would be inefficient to compute the value of the loss function and the value of the gradients independently, because doing so would lead to a lot of redundant computation between the two; the process would be almost twice as slow as computing them
jointly. To bypass this, you’ll set up a Python class named Evaluator that computes both the loss value and the gradients value at once, returns the loss value when called the first time, and caches the gradients for the next call.

In [46]:
### Setting up the gradient-descent process

with tf.compat.v1.Session() as sess:
    grads = tf.gradients(loss, combination_image)[0] # Get the gradients of the generated image wrt the loss
    fetch_loss_and_grads = K.function([combination_image], [loss, grads]) # Function to fetch the values of the current loss and the current gradients

    class Evaluator(object):
        def __init__(self):
            self.loss_value = None
            self.grads_values = None

        def loss(self, x):
            assert self.loss_value is None
            x = x.reshape((1, img_height, img_width, 3))
            outs = fetch_loss_and_grads([x])
            loss_value = outs[0]
            grad_values = outs[1].flatten().astype('float64')
            self.loss_value = loss_value
            self.grad_values = grad_values
            return self.loss_value

        def grads(self, x):
            assert self.loss_value is not None
            grad_values = np.copy(self.grad_values)
            self.loss_value = None
            self.grad_values = None
            return grad_values

Finally, you can run the gradient-ascent process using SciPy’s L-BFGS algorithm, saving the current generated image at each iteration of the algorithm.

In [47]:
### Style-transfer loop

from scipy.optimize import fmin_l_bfgs_b
import imageio
import time

result_prefix = 'style_transfer_result'
iterations = 20

x = np.array(evaluate_tensor(preprocess_image(target_image_path)), dtype=np.float32).flatten() # Initialize the generated image as the target image

for i in range(iterations):
    print('Start of iteration', i)
    start_time = time.time()
    evaluator = Evaluator()
    x, min_val, info = fmin_l_bfgs_b(evaluator.loss, x, fprime=evaluator.grads, maxfun=20) # Run L-BFGS optimization over the pixels of the generated image
    print('Current loss value:', min_val)
    img = deprocess_image(x.copy()) # Convert the generated image to a valid image format
    fname = result_prefix + '_at_iteration_%d.png' % i
    imageio.imsave(fname, img) # Save the generated image
    end_time = time.time()
    print('Image saved as', fname)
    print('Iteration %d completed in %ds' % (i, end_time - start_time))

Start of iteration 0
Current loss value: 5680098000.0
Image saved as style_transfer_result_at_iteration_0.png
Iteration 0 completed in 84s
Start of iteration 1
Current loss value: 1912387300.0
Image saved as style_transfer_result_at_iteration_1.png
Iteration 1 completed in 84s
Start of iteration 2
Current loss value: 1061592400.0
Image saved as style_transfer_result_at_iteration_2.png
Iteration 2 completed in 78s
Start of iteration 3
Current loss value: 760580900.0
Image saved as style_transfer_result_at_iteration_3.png
Iteration 3 completed in 74s
Start of iteration 4
Current loss value: 607574100.0
Image saved as style_transfer_result_at_iteration_4.png
Iteration 4 completed in 76s
Start of iteration 5
Current loss value: 506155840.0
Image saved as style_transfer_result_at_iteration_5.png
Iteration 5 completed in 78s
Start of iteration 6
Current loss value: 451163100.0
Image saved as style_transfer_result_at_iteration_6.png
Iteration 6 completed in 75s
Start of iteration 7
Current lo

Keep in mind that what this technique achieves is merely a form of image retexturing, or texture transfer. t works best with style-
reference images that are strongly textured and highly self-similar, and with content targets that don’t require high levels of detail in order to be recognizable. 